# New columns from old: assign, map and apply

Python stage, lesson P13. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** look values up with `map` and a dictionary, and see what `map` does with a key the dictionary lacks; add a column computed from other columns with `assign`; apply a small function to each value, and choose the vectorized form first.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## Unit 1: map, a lookup for a whole column

**Predict first.** Order 1107 has two lines, for product 5 and product 4. The products table says product 4 costs 4.50 and product 5 costs 5.00. What does `map` give for the two lines?

In [ ]:
price_of = dict(zip(products["product_id"], products["unit_price_usd"]))
order_1107 = order_items[order_items["order_id"] == 1107]
print(order_1107["product_id"].tolist())
print(order_1107["product_id"].map(price_of).tolist())

**Predict first.** A dictionary of store names was written before Harbor opened: it has stores 1 and 2 only. What does `map` give for an order of store 3, and for how many of the 1,800 orders? Choose between an error, a blank and something else.

In [ ]:
store_name_of = {1: "Old Town", 2: "Campus"}
names = orders["store_id"].map(store_name_of)
print(names.isna().sum(), names.notna().sum())

## Unit 2: assign, a new column from other columns

**Predict first.** The amount of a line is quantity times price minus discount. What shape will `lines` have, and what are the amounts of the two lines of order 1107 (quantities 2 and 1, discounts 0.50 and 0.50)? The total is for you to read.

In [ ]:
lines = order_items.assign(price=order_items["product_id"].map(price_of))
lines = lines.assign(amount=lines["quantity"] * lines["price"] - lines["discount_usd"])
print(order_items.shape, lines.shape)
print(lines[lines["order_id"] == 1107]["amount"].tolist())
print(lines["amount"].sum())

## Unit 3: apply, your own function for each value

**Predict first.** The function labels a quantity of 3 or more `bulk`, a quantity of 2 `double` and anything else `single`. Which label gets the most lines? The counts are for you to read.

In [ ]:
def size_label(quantity):
    if quantity >= 3:
        return "bulk"
    elif quantity == 2:
        return "double"
    return "single"


lines = lines.assign(size=lines["quantity"].apply(size_label))
print(lines["size"].value_counts())

**Predict first.** The same labels in one `np.where` line, without a function. Do the two columns agree on every line?

In [ ]:
size_np = np.where(lines["quantity"] >= 3, "bulk", np.where(lines["quantity"] == 2, "double", "single"))
print((lines["size"].to_numpy() == size_np).all())

## Unit 4: Check: the control total

**Predict first.** The control total of the amounts of all 3,287 lines is the one of lessons E15 and S7. What does the `amount` column add up to?

In [ ]:
print(lines["amount"].sum())

## Finish it

### Exercise 1: Finish it

**Task:** The happy-hour price is 80 percent of the list price. What do the 16 products cost at the happy-hour price, all together? Replace the first `None` with `products` plus a new column `happy_price`, made with `assign`, then set `answer_1` to the sum of that column.

**Expected:** one decimal number.

In [ ]:
happy = None  # replace None: products with a new column happy_price, 80 percent of unit_price_usd, made with assign
answer_1 = None  # then set answer_1 to the sum of the happy_price column

In [ ]:
check("Exercise 1", "answer_1", 60.0, traps={15.0: "15.0 is the discount, 20 percent. The happy-hour price is what is left: 80 percent of the list price."})

<details>
<summary>Hint</summary>

Unit 2 gives a table a new column with `assign(name=expression)`. The expression works on the whole column at once.

</details>

<details>
<summary>Solution</summary>

```python
happy = products.assign(happy_price=products["unit_price_usd"] * 0.8)
answer_1 = happy["happy_price"].sum()
```

The answer is 60.0: the 16 list prices add up to 75.0, and 80 percent of that is 60.0. Multiplying by 0.2 gives the discount, 15.0.

</details>

## On your own

### Exercise 2: On your own

**Task:** Recode the `status` column of `orders` with `map`: `completed` becomes `paid` and `canceled` becomes `void`. How many orders are `void`? Before you start, say in one sentence what kind of number the answer is. Write the code in the empty cell below and set `answer_2`.

**Expected:** one whole number.

In [ ]:
check("Exercise 2", "answer_2", 1, traps={0: "No order was recoded as void: a key of your dictionary does not match the values of the column. Print recoded.isna().sum() and check the spelling of the keys."})

<details>
<summary>Hint</summary>

Unit 1 turns each value of a column into another with `map` and a dictionary. A key that the column does not contain recodes nothing.

</details>

<details>
<summary>Solution</summary>

```python
status_label = {"completed": "paid", "canceled": "void"}
recoded = orders["status"].map(status_label)
answer_2 = (recoded == "void").sum()
```

The answer is 1: the one canceled order. A key spelled `cancelled` matches no value, so that order becomes `NaN` and the count is 0.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** A colleague labels the lines by quantity with the function below and checks that it runs without an error. But some lines get no label. Fix the function, then set `answer_3 = labels.notna().sum()`, the number of lines that have a label.

**Expected:** one whole number.

In [ ]:
def size_label(quantity):
    if quantity >= 3:
        return "bulk"
    elif quantity == 2:
        return "double"


labels = order_items["quantity"].apply(size_label)
answer_3 = None  # fix the function above, run the lines again, then set answer_3 = labels.notna().sum()

In [ ]:
check("Exercise 3", "answer_3", 3287, traps={1376: "Only the lines with a quantity of 2 or more have a label. What does a function give back when no branch runs?"})

<details>
<summary>Hint</summary>

Call the function by hand on a quantity of 1 and print what comes back. A function that reaches its end without a `return` gives `None`.

</details>

<details>
<summary>Solution</summary>

```python
def size_label(quantity):
    if quantity >= 3:
        return "bulk"
    elif quantity == 2:
        return "double"
    return "single"


labels = order_items["quantity"].apply(size_label)
answer_3 = labels.notna().sum()
```

The answer is 3287: every line has a label. Without the last `return`, the 1,911 lines with a quantity of 1 got `None`, and the count was 1376.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")